# Packed rows and document boundaries in NumPy

A GPT learns from rows of tokens of one fixed length, and documents come in every length. Padding
each document out to the longest one in its batch leaves most of a row's positions holding nothing
to learn from. The other way, the one GPT-2 used, is to join the documents into one long stream and
cut the stream into rows. You will write that cut yourself, then mark the predictions that land
between two documents.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

EOT = 1   # every document is closed by its end-of-text token
V = 9     # token ids 0 to 8

# Three documents, each closed by its end-of-text token.
docs = [[5, 3, 8, 2, 6, 4, 1], [2, 7, 7, 3, 1], [4, 1]]

stream = np.concatenate([np.asarray(d) for d in docs])
print(stream, "stream length:", len(stream))

## Part 1 (coding task): Packing documents into rows

Write `pack(docs, T)`. It joins the documents into one stream and cuts the stream into rows of
length $T$ with no pads at all. It returns `inputs` and `targets`, two integer arrays of shape
$(N, T)$, where `targets[i, t]` is the token that follows `inputs[i, t]` in the stream.

Read row by row, the targets must be the stream from its second token on, with nothing
skipped or repeated, up to the end of the last full row. Make $N$ as large as the stream
allows, and drop the tokens left over at the end.

In [ ]:
def pack(docs, T):
    """Join the documents into one stream and cut it into rows of inputs and next-token targets."""
    # TODO
    raise NotImplementedError

In [ ]:
X, Y = pack(docs, 4)
X, Y = np.asarray(X), np.asarray(Y)
assert X.shape == Y.shape, f"inputs and targets should have the same shape, got {X.shape} and {Y.shape}"
assert X.ndim == 2 and X.shape[1] == 4, f"each row should hold T = 4 tokens, got shape {X.shape}"
assert np.issubdtype(X.dtype, np.integer) and np.issubdtype(Y.dtype, np.integer), "inputs and targets should be integer token ids"
assert np.array_equal(X[:, 1:], Y[:, :-1]), "targets[i, t] should be the token right after inputs[i, t]"
assert X.shape[0] != 2, "the 14-token stream fills 3 rows of 4: rows of T + 1 tokens cut side by side never use the token at a row boundary as a target"
assert X.shape[0] == 3, f"the 14-token stream has 13 targets, enough for 3 rows of 4, got {X.shape[0]} rows"
assert np.array_equal(Y.ravel(), stream[1:13]), f"read row by row, the targets should be the stream from its second token on, got {Y.ravel()}"
try:
    X7, Y7 = pack(docs, 7)
except ValueError as e:
    raise AssertionError(f"pack(docs, 7) failed ({e}): 14 tokens hold only 13 targets, so they fill one row of 7, not two")
assert np.shape(X7) == (1, 7), f"with T = 7 the stream fills one row, since a second would need a 15th token as its last target; got shape {np.shape(X7)}"
X13, Y13 = pack(docs, 13)
assert np.array_equal(np.asarray(Y13).ravel(), stream[1:]), "with T = 13 one row uses the whole stream"

X6, Y6 = pack(docs, 6)
print(f"packed, T = 4:  {12} of the {len(stream) - 1} targets in the stream")
print(f"packed, T = 6:  {Y6.size} targets read row by row, {len(stream) - 1 - Y6.size} token dropped")
print("Part 1 passed")

## Part 2 (coding task): The predictions that cross a document

Every document in the stream is closed by its end-of-text token, but the rows are cut every $T$
tokens, so a document can stop in the middle of a row. The target that follows such a token is the
first token of the next document, and the model has never been shown a document beginning there:
the corpus has no such pair of neighbors.

Write `crosses_boundary(docs, T)`. Pack `docs` the way Part 1 does, then return a boolean array
of the shape of the packed targets, where entry `[i, t]` is True when `inputs[i, t]` is the last
token of a document, so that `targets[i, t]` is the first token of the next one. Every other entry
is False. A document's end at the very end of the stream is not marked, since no target follows
it.

In [ ]:
def crosses_boundary(docs, T):
    """True where a packed target starts a new document: the input before it ended one."""
    # TODO
    raise NotImplementedError

In [ ]:
X, Y = pack(docs, 4)
M = np.asarray(crosses_boundary(docs, 4))
assert M.shape == np.asarray(Y).shape, f"the mask should have the shape of the packed targets {np.asarray(Y).shape}, got {M.shape}"
assert M.dtype == bool, f"the mask should hold booleans, got {M.dtype}"
assert M.sum() != 3, "the rows are cut every 4 tokens, not at the documents: marking each row's last column would put 3 marks here"
assert not np.all(M[:, -1]), "the rows are cut every 4 tokens, not at the documents: marking each row's last column would put 3 marks here"
assert M.sum() != 1, "a document's first token is an input like any other, so it should not be marked: only the token before it, which ends the previous document"
assert M.sum() == 2, f"the documents end at stream positions 6, 11 and 13, and this cut reads the 12 inputs at 0 to 11, so two targets cross a document; got {M.sum()}"
assert np.array_equal(np.argwhere(M), np.array([[1, 2], [2, 3]])), f"stream positions 6 and 11 sit at row 1 column 2 and row 2 column 3; got {np.argwhere(M).tolist()}"
assert np.array_equal(np.asarray(X)[M], [EOT, EOT]), f"a marked input ends a document, so both should be the end-of-text token; got {np.asarray(X)[M].tolist()}"
assert np.array_equal(np.asarray(Y)[M], [docs[1][0], docs[2][0]]), f"a marked target starts the next document, so they should be {docs[1][0]} and {docs[2][0]}; got {np.asarray(Y)[M].tolist()}"

M13 = np.asarray(crosses_boundary(docs, 13))
assert np.array_equal(np.argwhere(M13), np.array([[0, 6], [0, 11]])), f"with T = 13 the whole stream is one row, so the same two ends sit at columns 6 and 11; got {np.argwhere(M13).tolist()}"
M5 = np.asarray(crosses_boundary(docs, 5))
assert np.array_equal(np.argwhere(M5), np.array([[1, 1]])), f"with T = 5 only the 10 inputs at 0 to 9 are read, so the second document's end at 11 is never an input; got {np.argwhere(M5).tolist()}"

M2 = np.asarray(crosses_boundary([[5, 1, 3], [2, 1]], 2))
assert np.array_equal(M2, np.array([[False, False], [True, False]])), f"an end-of-text token inside a document is not a document end: only a document's last token is marked; got {M2.tolist()}"

print(f"T = 4:  {int(M.sum())} of {M.size} targets start a document, {M.sum() / M.size:.0%} of the loss")
print("Part 2 passed")

## Solution

### Part 1: Packing documents into rows

In [ ]:
def pack(docs, T):
    """Join the documents into one stream and cut it into rows of inputs and next-token targets."""
    stream = np.concatenate([np.asarray(d) for d in docs])
    n = (len(stream) - 1) // T                     # each row needs T inputs and one more token for its last target
    inputs = stream[: n * T].reshape(n, T)
    targets = stream[1 : n * T + 1].reshape(n, T)
    return inputs, targets

**Why it works.** Row $i$ reads the stream from position $iT$, and its targets start one
later, at $iT + 1$. So row $i + 1$ starts on the last target of row $i$, and consecutive rows
overlap by one token. That overlap keeps every token after the first a target: rows of $T + 1$
tokens cut side by side would lose the token at each boundary. A stream of $L$ tokens has $L - 1$
targets, so it fills $\lfloor (L - 1)/T \rfloor$ rows.

### Part 2: The predictions that cross a document

In [ ]:
def crosses_boundary(docs, T):
    """True where a packed target starts a new document: the input before it ended one."""
    inputs, _ = pack(docs, T)
    stream = np.concatenate([np.asarray(d) for d in docs])
    ends = np.cumsum([len(d) for d in docs]) - 1                # where each document ends in the stream
    row = np.arange(inputs.shape[0]).reshape(-1, 1) * T         # the stream position each row starts at
    position = row + np.arange(inputs.shape[1])                # the stream position of each input
    return np.isin(position, ends)

**Why it works.** An input at row $i$, column $t$ is the token at stream position $iT + t$,
so a document that ends at stream position $p$ puts its end-of-text token at row
$\lfloor p / T \rfloor$, column $p \bmod T$. The three documents end at 6, 11 and 13, and the
cut with $T = 4$ reads the 12 inputs at 0 to 11, so two of the three ends are inputs: at $(1, 2)$
and $(2, 3)$. Their targets are the 2 and the 4, the first tokens of the second and third
documents, which is 2 of the 12 predictions in this batch: the model is asked to write a fresh
document starting one token after a document ended, a pair the corpus never contains. A pipeline
either accepts those terms, and the model learns that a document can start right after an
end-of-text token, or leaves them out of the loss as the padding mask leaves out a pad target.